# Week 1 Day 5 Lab 2 (free version) - Harness Engineering

## Deep Agents works the queue on its own

Deep Agents is LangChain's ready-made harness. It's built on `create_agent`, the same function we used on Day 2, with middleware added for file tools, subagents, a to-do list and summarizing a long conversation. It runs on any model LangChain supports, so we can use OpenRouter.

This is the same job as Lab 1: a folder with three claims from the queue, the policy wording and the claims manual, and the instruction to decide every claim. We don't write any tools.

## 4 parts:

PART 1: The claims desk

PART 2: Setting up the harness

PART 3: Working the queue

PART 4: How it did

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This version uses the free version of dots.3 Note Preview on OpenRouter, so it costs nothing. We name one free model because <code>openrouter/free</code> picks a different model for nearly every call, and some of them can't keep a job like this on track.<br/><br/>
            An OpenRouter account without credit can make 50 free requests a day. Three claims take about 35 requests and a few minutes, so you can run this once a day. If you have bought credit, the main version in <a href="week1day5_lab2.ipynb">week1day5_lab2.ipynb</a> costs about 3 cents.
            </span>
        </td>
    </tr>
</table>

In [2]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import ModelRetryMiddleware, TodoListMiddleware
from langchain_openai import ChatOpenAI

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


## Part 1: The claims desk

The claims desk is the same as in Lab 1. We use the first three of the five claims marked for the tiny version of the queue: an approval, a decline and a referral.

In [4]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [6]:
claims, labels = get_sample_data(tiny=True)
claims = claims[:5]
labels = {claim["claim_id"]: labels[claim["claim_id"]] for claim in claims}

for claim in claims:
    print(claim["claim_id"], claim["summary"])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


In [7]:
DESK = Path("lab2free_desk")


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (DESK / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (DESK / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

## Part 2: Setting up the harness

The instructions are the same as in Lab 1, apart from calling the task list a to-do list, which is Deep Agents' name for it.

In [8]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler subagent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Deep Agents</h2>
            <span style="color:#118AB2;"><code>create_deep_agent</code> gives the agent file tools (<code>ls</code>, <code>read_file</code>, <code>write_file</code>, <code>edit_file</code>, <code>glob</code>, <code>grep</code>) and <code>task</code>, which hands work to a subagent.<br/><br/>
            The to-do list is an extra since version 0.7, so we add it with LangChain's <code>TodoListMiddleware</code>, which gives the agent a <code>write_todos</code> tool.<br/><br/>
            <code>FilesystemBackend</code> points the file tools at the claims desk. With <code>virtual_mode=True</code> the agent sees the desk as <code>/</code> and can't reach anything outside it.
            </span>
        </td>
    </tr>
</table>

In [9]:
llm = ChatOpenAI(
    model="dots-studio/dots-3-note-preview:free",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    timeout=120,
    max_retries=5,
)

Free models are sometimes busy and a call can fail. LangChain's `ModelRetryMiddleware` waits a moment and tries a failed call again. We give it to the main agent and to the subagent.

In [10]:
claim_handler = {
    "name": "claim-handler",
    "description": "Decides one claim and writes its decision file. Give it the claim id.",
    "system_prompt": HANDLER_INSTRUCTIONS,
    "middleware": [ModelRetryMiddleware(max_retries=3)],
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware(), ModelRetryMiddleware(max_retries=3)],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
)

## Part 3: Working the queue

As the agent works we print each step and count the model calls. The stream gives us an update after each step of the agent, and the updates from the `model` step hold the model's replies. The subagent's steps are indented.

In [11]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call["args"]
    if not isinstance(args, dict):  # a free model sometimes sends its arguments as one string
        return tool_call["name"]
    if "file_path" in args or "path" in args:
        return f"{tool_call['name']} {args.get('file_path') or args.get('path')}"
    if tool_call["name"] == "task":
        return f"task: {args.get('description', '').splitlines()[0][:80]}"
    if tool_call["name"] == "write_todos":
        return "write_todos: " + ", ".join(f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", []))
    return tool_call["name"]

In [12]:
calls = 0

for namespace, update in agent.stream(
    {"messages": [{"role": "user", "content": "Please work the queue."}]},
    stream_mode="updates",
    subgraphs=True,
    config={"recursion_limit": 200},
):
    if "model" in update:
        indent = "      " if namespace else ""
        for message in update["model"]["messages"]:
            calls += 1
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())

print(f"\nFinished after {calls} model calls")

ls .
I'll start by exploring the folder structure to understand what we're working with.
ls /claims
ls /policies
ls /customers
ls /decisions
read_file /policy_wording.md
read_file /claims_manual.md
read_file /company.json
Let me read the key reference documents first.
read_file /policy_wording.md
read_file /claims_manual.md
Let me read the rest of the policy wording and claims manual.
read_file /claims/CLM-1007.json
read_file /claims/CLM-1015.json
read_file /claims/CLM-1017.json
read_file /claims/CLM-1020.json
read_file /claims/CLM-1021.json
Now let me read all the claims in the queue.
read_file /policies/POL-2007.json
read_file /policies/POL-2015.json
read_file /policies/POL-2017.json
read_file /policies/POL-2020.json
read_file /policies/POL-2021.json
read_file /customers/CUS-3007.json
read_file /customers/CUS-3015.json
read_file /customers/CUS-3017.json
read_file /customers/CUS-3020.json
read_file /customers/CUS-3021.json
Now let me look up the policies and customers for each claim.


## Part 4: How it did

We score the decision files with the same test as in Lab 1.

In [13]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(float(answer.get("payout") or 0) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id,
        "decision": answer.get("decision"),
        "payout": answer.get("payout"),
        "handler's decision": label["decision"],
        "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

0 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,Approve,50.0,approve,50.0,False
1,CLM-1015,Decline,0.0,decline,0.0,False
2,CLM-1017,Refer,0.0,refer,0.0,False
3,CLM-1020,Approve,240.0,approve,240.0,False
4,CLM-1021,Approve,1053.0,approve,825.0,False


And here's the note it left for the handler.

In [14]:
display(Markdown((DESK / "handover.md").read_text(encoding="utf-8")))

# Handover — Claims Queue Worked

All 5 claims in the queue have been decided and decision records written to `/decisions/`.

## Summary of decisions

| Claim | Customer | Decision | Payout | Notes |
|---|---|---|---|---|
| CLM-1007 | Greg Kowalski | **Approve** | $50.00 | Phone dropped in hotel shower (accidental damage). Economy per-item limit $150, deductible $100 → $50. No Electronics Plus add-on. |
| CLM-1015 | Kate Callister | **Decline** | $0.00 | Trip interruption after leaving a child at home. Not one of the five covered reasons under Section 2 ("We do not pay for cancellation or interruption for any other reason"). |
| CLM-1017 | Emily Nguyen | **Refer** | $0.00 | Backpack snatched in Naples. Customer says a police report was made but has not provided it. Referred with a 30-day document request for the police report. If provided, payout would be ~$420. |
| CLM-1020 | Nadia Rahman | **Approve** | $240.00 | Prescription sunglasses taken by a monkey at Uluwatu Temple. Treated as accidental loss per the wild animals interpretation note (no police report needed); licensed guide's letter accepted as evidence. First plan, no deductible. |
| CLM-1021 | Camille Fontaine | **Approve** | $1,053.00 | Tote bag theft in Buenos Aires. Police report on file within 24 hours; Electronics Plus add-on raises per-item limit for electronics to $1,200. Eligible $1,103 − $50 deductible = $1,053. |

## Items requiring your attention

1. **CLM-1021 — sign-off required.** The payout of $1,053.00 exceeds the $1,000 payout approval limit. The decision is recorded as Approve with the payout, but payment is waiting for a claims handler's sign-off. Please check the decision and calculation and record your name against the payment. Calculation: headphones $429 + e-reader $180 + game console $399 + tote $95 = $1,103 eligible; minus $50 Premium deductible = $1,053; under the $1,500 benefit maximum.

2. **CLM-1017 — open, awaiting document.** The claim is referred pending the police report from the Carabinieri in Naples. If the report arrives within 30 days, work it for an expected payout of ~$420 (Economy limits: tablet $450→$150, rain jacket $160→$150, backpack $90, sunglasses $130 = $520 eligible, minus $100 deductible). If the report does not arrive or shows the theft was never reported, decline under Section 5 (theft condition not met).

## Things worth knowing

- No claim in this queue hit the repeat-claims rule (3 claims in 12 months) — all customers had empty claim histories.
- No claim raised honesty concerns requiring referral to Special Investigations.
- CLM-1020 is a good example of the wild animals note in the claims manual: the temple police post would not take a report, but the licensed guide's letter is acceptable evidence and no police report is required for a wild animal loss.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Build or buy</h2>
            <span style="color:#118AB2;">Deep Agents is open source and runs on any model, so you can pick a cheap one and read exactly what the harness does. The Claude Agent SDK gives you Claude Code's harness, tuned for Claude models.<br/><br/>
            Next we build the same pieces into UnderCover ourselves: a to-do list, tools to read the manual, and a subagent for each claim.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Look back through the steps and find where the agent used its to-do list and where it handed work to the subagent.<br/><br/>
            Then try other free models from OpenRouter's list, or <code>openrouter/free</code>, and see which ones can work the queue and how their answers compare.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>